<a href="https://colab.research.google.com/github/jenitchadha/SDS-510/blob/main/MOD5_Essentials.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 5 ESSENTIALS PROJECT

I will be adding to the basics project. Please scroll below to see the things I have added for essentials.

In [2]:
### downloading the jeopardy dataset
import html
from urllib.request import urlretrieve

url = "https://www.dropbox.com/s/x7z8bk4vgowohp9/jeopardy.json?dl=1"
urlretrieve(url, "jeopardy.json")

('jeopardy.json', <http.client.HTTPMessage at 0x79ba6a295490>)

In [3]:
## creating a df
import pandas as pd

df = pd.read_json("jeopardy.json")

print(f"## of rows: {len(df):,}")
df[["question", "value", "round"]].head()

## of rows: 216,930


,question,value,round
0,"'For the last 8 years of his life, Galileo was...",$200,Jeopardy!
1,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jeopardy!
2,'The city of Yuma in this state has a record a...,$200,Jeopardy!
3,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,Jeopardy!
4,"'Signer of the Dec. of Indep., framer of the C...",$200,Jeopardy!


In [4]:
## clearing up the dollar values under "value"
## here I will be adding a new column called "points"
## to store my processed data
df["points"] = pd.to_numeric(df["value"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce")

df = df.dropna(subset=["question", "points"])
df = df.loc[df["points"] > 0].copy()
df["points"] = df["points"].astype(int)

print(f"Rows after cleaning: {len(df):,}")
df[["question", "value", "points"]].head()

Rows after cleaning: 213,296


,question,value,points
0,"'For the last 8 years of his life, Galileo was...",$200,200
1,'No. 2: 1912 Olympian; football star at Carlis...,$200,200
2,'The city of Yuma in this state has a record a...,$200,200
3,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,200
4,"'Signer of the Dec. of Indep., framer of the C...",$200,200


In [5]:
## now I will be doing something similar to the question
## and remove html tags and links, convert all the text into
## lowercase, and remove any extra spaces
## I will create a new "clean_question" column

df["clean_question"] = (df["question"]
    .map(html.unescape)
    .str.replace(r"<[^>]+>", " ", regex=True)
    .str.replace(r"https?://\S+", " ", regex=True)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())

df = df.loc[df["clean_question"].str.contains(r"\w{2,}")].copy()

print(f"Rows after cleaning the text: {len(df):,}")
df[["question", "clean_question"]].head()

Rows after cleaning the text: 213,235


,question,clean_question
0,"'For the last 8 years of his life, Galileo was...","'for the last 8 years of his life, galileo was..."
1,'No. 2: 1912 Olympian; football star at Carlis...,'no. 2: 1912 olympian; football star at carlis...
2,'The city of Yuma in this state has a record a...,'the city of yuma in this state has a record a...
3,"'In 1963, live on ""The Art Linkletter Show"", t...","'in 1963, live on ""the art linkletter show"", t..."
4,"'Signer of the Dec. of Indep., framer of the C...","'signer of the dec. of indep., framer of the c..."


In [6]:
### So we need to create a binary classifier with
### high and low values based on the points in each row.
### I am using a cut off of $600.

cutoff = 600

df["label"] = (df["points"] > cutoff).map({
    True: "high value",
    False: "low value"})

labels_per_clue = df.groupby("clean_question")["label"].transform("nunique")
df = df.loc[labels_per_clue == 1].copy()

df = df.drop_duplicates(subset="clean_question")

print(df["label"].value_counts())
print(f"Total clues to be split: {len(df):,}")

label
low value     119010
high value     92929
Name: count, dtype: int64
Total clues to be split: 211,939


In [7]:
## splitting training and test sets
from sklearn.model_selection import train_test_split

X = df["clean_question"]  # Clue text the model reads
y = df["label"]           # Correct label for each clue

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print(f"Training set: {len(X_train):,}")
print(f"Test set: {len(X_test):,}")

Training set: 169,551
Test set: 42,388


In [8]:
## training the naive bayes classifier
## Naive Bayes learns which words are more common in each
## label using the training clues, then predicts a label
## for each test clue.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    CountVectorizer(),
    MultinomialNB())

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"Predictions made: {len(predictions):,}")
print("First five predictions:", predictions[:5])

Predictions made: 42,388
First five predictions: ['low value' 'low value' 'high value' 'low value' 'low value']


In [9]:
## calculating accuracy

from pathlib import Path
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

most_common_label = y_train.value_counts().idxmax()
baseline_accuracy = (y_test == most_common_label).mean()

results = (
    f"Test clues: {len(y_test):,}\n"
    f"Naive Bayes accuracy: {accuracy:.2%}\n"
    f"Baseline accuracy (always predict '{most_common_label}'): "
    f"{baseline_accuracy:.2%}\n"
    f"Difference: {(accuracy - baseline_accuracy) * 100:+.2f} percentage points\n")

print(results)
Path("jeopardy_results.txt").write_text(results, encoding="utf-8")

Test clues: 42,388
Naive Bayes accuracy: 58.07%
Baseline accuracy (always predict 'low value'): 56.15%
Difference: +1.92 percentage points



139

The model got 58.07% of the test clues right. Guessing “low value” every time got 56.15% right. Thus, the words in the clues helped the model a little.

## THE ESSENTIALS PROJECT

Ok, so for this project, my two other approaches beyond Naive Bayes would be logistic regression and decision trees.

Logistic regression will give words weights and combine them to estimate the probability that a clue is of high value.

On the other hand, decion tree will learns a series of yes-or-no questions about the clue’s words.

I have done this kind of project in R, but it will be slightly challenging to do in python. But I am ready for this challenge. Not quite sure I will be able to make it till the end, but will try!

In [10]:
## logistic regression
## notes - tf-idf will turn the clue words into numbers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

logistic_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=500))

logistic_model.fit(X_train, y_train)
logistic_accuracy = logistic_model.score(X_test, y_test)

print(f"Logistic Regression test accuracy: {logistic_accuracy:.2%}")
print(f"Naive Bayes test accuracy: {accuracy:.2%}")

Logistic Regression test accuracy: 58.50%
Naive Bayes test accuracy: 58.07%


Logistic Regression scored 58.50%, which is only 0.43 percentage points higher than Naive Bayes.

In [12]:
## decision trees
## I will limit how many decisions this can make to 20
## will use max_depth=20 for this
from sklearn.tree import DecisionTreeClassifier

tree_model = make_pipeline(
    TfidfVectorizer(),
    DecisionTreeClassifier(max_depth=20, random_state=42))

tree_model.fit(X_train, y_train)
tree_accuracy = tree_model.score(X_test, y_test)

print(f"Decision Tree test accuracy: {tree_accuracy:.2%}")
print(f"Logistic Regression test accuracy: {logistic_accuracy:.2%}")
print(f"Naive Bayes test accuracy: {accuracy:.2%}")
print(f"Always guess low: {baseline_accuracy:.2%}")

Decision Tree test accuracy: 57.13%
Logistic Regression test accuracy: 58.50%
Naive Bayes test accuracy: 58.07%
Always guess low: 56.15%


From the above output, logistic Regression scored highest at 58.50%, which is 2.35 percentage points better than Always Guess Low.

In [20]:
### printing the decision tree
### I kind of struggled with it and needed alot of assistance.
from pathlib import Path
from sklearn.tree import export_text

vectorizer = tree_model.named_steps["tfidfvectorizer"]
tree = tree_model.named_steps["decisiontreeclassifier"]
words = vectorizer.get_feature_names_out().tolist()

print(export_text(tree, feature_names=words, max_depth=2, decimals=4))

# Saving the tree.
full_tree = export_text(
    tree,
    feature_names=words,
    max_depth=tree.get_depth(),
    decimals=6)

Path("jeopardy_decision_tree.txt").write_text(full_tree, encoding="utf-8")


|--- clue <= 0.0838
|   |--- these <= 0.1291
|   |   |--- this <= 0.0626
|   |   |   |--- truncated branch of depth 18
|   |   |--- this >  0.0626
|   |   |   |--- truncated branch of depth 18
|   |--- these >  0.1291
|   |   |--- name <= 0.1448
|   |   |   |--- truncated branch of depth 18
|   |   |--- name >  0.1448
|   |   |   |--- truncated branch of depth 18
|--- clue >  0.0838
|   |--- crew <= 0.0403
|   |   |--- clue <= 0.1807
|   |   |   |--- truncated branch of depth 18
|   |   |--- clue >  0.1807
|   |   |   |--- truncated branch of depth 13
|   |--- crew >  0.0403
|   |   |--- zoo <= 0.2249
|   |   |   |--- truncated branch of depth 18
|   |   |--- zoo >  0.2249
|   |   |   |--- truncated branch of depth 3



554440